In [1]:
# 1. Set QUICK_TRIAL = True for a ~20 minute test of the whole pipeline on a few
#    minutes of audio. Set it to False for the real run, then use Save Version.
QUICK_TRIAL = False
HOURS_SCALE = 0.05 if QUICK_TRIAL else 1.0

In [2]:
import glob, os, shutil, subprocess, time
REPO = "https://github.com/23f2001033/IITH-Intern-Assignment-.git"
PROJECT = "/tmp/project"
shutil.rmtree(PROJECT, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", REPO, PROJECT], check=True)
print(subprocess.run(["git", "-C", PROJECT, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
subprocess.run("pip install -q pyworld tgt g2p-en inflect unidecode", shell=True, check=True)

Cloning into '/tmp/project'...


5aac935 Kaggle Notebook | IIIT-H | Version 2



CompletedProcess(args='pip install -q pyworld tgt g2p-en inflect unidecode', returncode=0)

In [3]:
# 2. Install Montreal Forced Aligner
MFA = "/tmp/bin/micromamba run -r /tmp/mamba -n mfa mfa"
if not os.path.exists("/tmp/mamba/envs/mfa"):
    subprocess.run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /tmp bin/micromamba", shell=True, check=True)
    subprocess.run("/tmp/bin/micromamba create -y -q -r /tmp/mamba -n mfa -c conda-forge montreal-forced-aligner", shell=True, check=True)
print(subprocess.run(MFA + " version", shell=True, capture_output=True, text=True).stdout)

3.4.3



In [4]:
# 3. Download both datasets (your HF_TOKEN secret must be attached to this notebook)
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
subprocess.run("pip install -q -U huggingface_hub", shell=True, check=True)
for repo, name in (("ai4bharat/Svarah", "svarah"), ("SPRINGLab/IndicTTS_Telugu", "telugu")):
    if not glob.glob("/tmp/raw/{}/data/*.parquet".format(name)):
        subprocess.run("hf download {} --repo-type dataset --local-dir /tmp/raw/{}".format(repo, name), shell=True, check=True)
del os.environ["HF_TOKEN"]

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Fetching 5 files: 100%|██████████| 5/5 [00:07<00:00,  1.42s/it]


✓ Downloaded
  path: /tmp/raw/svarah


Fetching 6 files:  67%|██████▋   | 4/6 [00:22<00:12,  6.17s/it]

✓ Downloaded
  path: /tmp/raw/telugu


Fetching 6 files: 100%|██████████| 6/6 [00:32<00:00,  5.35s/it]


In [5]:
# 4. Prepare, align and extract features for both languages at the same time.
#    Each language gets its own MFA folder because both corpora are called raw_data.
os.makedirs("/kaggle/working/logs", exist_ok=True)
jobs = {}
for language, corpus in (("English", "/tmp/raw/svarah"),):
    work = "/tmp/tts/" + language
    shutil.rmtree(work, ignore_errors=True)
    shutil.rmtree("/tmp/mfa_" + language, ignore_errors=True)
    command = [
        "python", "{}/{}/preprocess.py".format(PROJECT, language), "--stage", "all",
        "--corpus_path", corpus, "--work_dir", work, "--mfa_command", MFA,
        "--jobs", "4", "--hours_scale", str(HOURS_SCALE),
    ]
    log = open("/kaggle/working/logs/prepare_{}.log".format(language), "w")
    env = dict(os.environ, MFA_ROOT_DIR="/tmp/mfa_" + language)
    jobs[language] = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, env=env)

while any(p.poll() is None for p in jobs.values()):
    time.sleep(120)
    print(time.strftime("%H:%M"), {k: ("running" if p.poll() is None else p.returncode) for k, p in jobs.items()}, flush=True)
print({k: p.returncode for k, p in jobs.items()})
for language in jobs:
    print("=" * 30, language)
    print(open("/kaggle/working/logs/prepare_{}.log".format(language), errors="replace").read()[-2500:])

01:00 {'English': 'running'}
01:02 {'English': 'running'}
01:04 {'English': 'running'}
01:06 {'English': 'running'}
01:08 {'English': 'running'}
01:10 {'English': 'running'}
01:12 {'English': 'running'}
01:14 {'English': 0}
{'English': 0}
============================== English
<01:43,  2.36s/it]
100%|██████████| 112/112 [04:22<00:00,  2.34s/it]
Computing statistic quantities ...
{'hours_after_silence_trim': {'train': 4.846359284454522, 'val': 0.9805819098009574}, 'utterances': {'train': 3303, 'val': 692}, 'skipped': {'no_alignment': 0, 'unusable': 1, 'unknown_word': 0, 'too_long': 0}, 'speakers': 112}



In [6]:
# 5. Pack what training needs into one file per language
for language, job in jobs.items():
    if job.returncode != 0:
        print("SKIPPED", language, "- it failed, see its log above")
        continue
    work = "/tmp/tts/" + language
    subprocess.run(["tar", "-cf", "/kaggle/working/{}_preprocessed.tar".format(language), "-C", work, "preprocessed_data"], check=True)
    os.makedirs("/kaggle/working/reports/" + language, exist_ok=True)
    shutil.copy(work + "/preprocessed_data/filelists/data_report.json", "/kaggle/working/reports/" + language)
    shutil.copy(work + "/preprocessed_data/feature_summary.json", "/kaggle/working/reports/" + language)
    for name in ("train.ids", "val.ids"):
        shutil.copy(work + "/preprocessed_data/filelists/" + name, "/kaggle/working/reports/" + language)
    print(language, open(work + "/preprocessed_data/feature_summary.json").read())
subprocess.run("ls -la /kaggle/working; du -sh /kaggle/working/*", shell=True)

English {
  "hours_after_silence_trim": {
    "train": 4.846359284454522,
    "val": 0.9805819098009574
  },
  "utterances": {
    "train": 3303,
    "val": 692
  },
  "skipped": {
    "no_alignment": 0,
    "unusable": 1,
    "unknown_word": 0,
    "too_long": 0
  },
  "speakers": 112
}
total 615656
drwxr-xr-x 4 root root      4096 Nov  6 01:14 .
drwxr-xr-x 8 root root      4096 Nov  6 00:56 ..
-rw-r--r-- 1 root root 630394880 Nov  6 01:14 English_preprocessed.tar
drwxr-xr-x 2 root root      4096 Nov  6 00:58 logs
-rw-r--r-- 1 root root     17750 Nov  6 01:14 __notebook__.ipynb
drwxr-xr-x 3 root root      4096 Nov  6 01:14 reports
602M	/kaggle/working/English_preprocessed.tar
20K	/kaggle/working/__notebook__.ipynb
48K	/kaggle/working/logs
140K	/kaggle/working/reports


CompletedProcess(args='ls -la /kaggle/working; du -sh /kaggle/working/*', returncode=0)